# SIC 2026 · Inteligencia Artificial
## Clase 2: NumPy, pandas y primer modelo de machine learning

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

En esta clase vamos a repasar las dos librerías que usaremos en todo el curso, NumPy y pandas, y vamos a entrenar nuestro primer modelo de machine learning con datos reales.

| Parte | Tema | Tiempo aproximado |
|---|---|---|
| 0 | Cómo usar este notebook | 10 min |
| 1 | NumPy: cálculo con arreglos | 50 min |
| 2 | pandas: trabajar con tablas de datos | 80 min |
| 3 | Introducción a machine learning | 80 min |
| 4 | Desafío final y guardar en GitHub | 20 min |

**Cómo trabajar:** lee cada explicación, ejecuta las celdas de ejemplo y resuelve los ejercicios marcados con `# TU CÓDIGO AQUÍ`. Después de cada ejercicio hay una celda de verificación que te dice si tu respuesta es correcta. Las soluciones están al final del notebook; intenta resolver cada ejercicio antes de mirarlas.

---
# Parte 0. Cómo usar este notebook

Un notebook de Colab está hecho de **celdas**. Hay dos tipos:

- **Celdas de texto** como esta, con explicaciones.
- **Celdas de código**, que se ejecutan con `Shift + Enter` o con el botón de reproducir a la izquierda.

Las celdas comparten memoria: una variable creada en una celda se puede usar en las siguientes. Por eso es importante ejecutarlas **en orden**. Si algo falla de forma extraña, usa *Entorno de ejecución > Reiniciar y ejecutar todo*.

**Antes de empezar**, guarda tu propia copia: *Archivo > Guardar una copia en Drive*. Así tus cambios no se pierden.

Ejecuta la siguiente celda. Importa las librerías y define una función `revisar` que usaremos para verificar los ejercicios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("NumPy", np.__version__, "| pandas", pd.__version__)

---
# Parte 1. NumPy: cálculo con arreglos

**NumPy** es la librería base del cálculo numérico en Python. Su objeto central es el **arreglo** (`ndarray`): una colección de números del mismo tipo, organizada en una o más dimensiones.

¿Por qué no usar listas de Python? Por dos razones:

1. **Velocidad.** Las operaciones de NumPy están escritas en C y procesan millones de números en milisegundos.
2. **Operaciones vectorizadas.** Puedes operar sobre todo el arreglo de una vez, sin escribir ciclos `for`.

pandas, scikit-learn y TensorFlow están construidos sobre NumPy, así que todo lo que aprendas aquí lo vas a reutilizar.

## 1.1 Crear arreglos

In [ ]:
# Desde una lista
a = np.array([3, 7, 1, 9, 4])
print("a =", a)

# Secuencias: np.arange(inicio, fin, paso). El fin NO se incluye
b = np.arange(0, 10, 2)
print("b =", b)

# n números equiespaciados entre dos valores (ambos incluidos)
c = np.linspace(0, 1, 5)
print("c =", c)

# Arreglos de ceros y de unos con una forma dada
print(np.zeros(3))
print(np.ones((2, 3)))

## 1.2 Forma, dimensiones y tipo

Todo arreglo tiene:

- `shape`: su forma, como tupla. Un vector de 5 elementos tiene forma `(5,)`; una matriz de 2 filas y 3 columnas, `(2, 3)`.
- `ndim`: el número de dimensiones.
- `dtype`: el tipo de dato de sus elementos (`int64`, `float64`, `bool`, etc.).

`reshape` reorganiza los mismos datos en otra forma, siempre que el total de elementos coincida.

In [ ]:
m = np.arange(12)
print(m, "| shape:", m.shape, "| ndim:", m.ndim, "| dtype:", m.dtype)

M = m.reshape(3, 4)   # 3 filas, 4 columnas: 3 x 4 = 12 elementos
print(M)
print("shape:", M.shape, "| ndim:", M.ndim)

## 1.3 Operaciones vectorizadas

Las operaciones aritméticas se aplican **elemento a elemento**. Compara las dos formas de sumar 10 a cada número:

In [ ]:
precios = np.array([1990, 3490, 590, 12990])

# Con un ciclo (estilo listas de Python)
con_ciclo = []
for p in precios:
    con_ciclo.append(p + 10)

# Vectorizado (estilo NumPy)
vectorizado = precios + 10

print(con_ciclo)
print(vectorizado)

# También entre arreglos del mismo tamaño
cantidades = np.array([2, 1, 5, 1])
print("Total por producto:", precios * cantidades)
print("Total de la compra:", (precios * cantidades).sum())

La diferencia de velocidad se nota con muchos datos. Ejecuta esta celda:

In [ ]:
import time
x = np.random.rand(5_000_000)

t0 = time.time(); r1 = [v * 2 for v in x]; t1 = time.time()
r2 = x * 2; t2 = time.time()
print(f"Ciclo de Python: {t1 - t0:.3f} s")
print(f"NumPy vectorizado: {t2 - t1:.3f} s")

## 1.4 Agregaciones y el parámetro `axis`

Las funciones de agregación resumen un arreglo: `sum`, `mean`, `min`, `max`, `std` (desviación estándar), `argmax` (posición del máximo).

En matrices, `axis` indica en qué dirección resumir:

- `axis=0`: recorre las filas hacia abajo, y da **un resultado por columna**.
- `axis=1`: recorre las columnas hacia la derecha, y da **un resultado por fila**.

In [ ]:
# Ventas de 3 tiendas (filas) durante 4 semanas (columnas)
ventas = np.array([[120, 135, 150, 110],
                   [ 90,  95, 100, 105],
                   [200, 180, 210, 190]])

print("Total general:", ventas.sum())
print("Total por semana (axis=0):", ventas.sum(axis=0))
print("Promedio por tienda (axis=1):", ventas.mean(axis=1))
print("Tienda con más ventas totales:", ventas.sum(axis=1).argmax())

## 1.5 Indexación, cortes y máscaras booleanas

- `arr[i]` toma un elemento; `arr[inicio:fin]` toma un corte (el fin no se incluye).
- En matrices: `M[fila, columna]`. Los dos puntos `:` significan "todas".
- Una **máscara booleana** es un arreglo de `True`/`False` que sirve para filtrar. Este es uno de los patrones más usados en análisis de datos.

In [ ]:
print("Primera tienda:", ventas[0])
print("Semana 3 de la tienda 2:", ventas[1, 2])
print("Todas las tiendas, semanas 1 y 2:\n", ventas[:, :2])

notas = np.array([5.5, 3.8, 6.2, 4.0, 2.9, 6.8])
mascara = notas >= 4.0
print("Máscara:", mascara)
print("Notas aprobadas:", notas[mascara])
print("Cantidad de aprobados:", mascara.sum())   # True cuenta como 1

## 1.6 Broadcasting

Cuando operas arreglos de formas distintas, NumPy "estira" el más pequeño para que calcen, si es posible. Esto se llama **broadcasting**. Ya lo usaste al sumar un número a un arreglo. También funciona entre una matriz y un vector:

In [ ]:
# Restar a cada columna su promedio (centrar los datos), algo muy común en ML
promedio_semana = ventas.mean(axis=0)          # forma (4,)
centradas = ventas - promedio_semana           # (3, 4) - (4,) -> (3, 4)
print(centradas)
print("Nuevo promedio por columna:", centradas.mean(axis=0))

## 1.7 Números aleatorios

Para simular datos y para que los modelos sean reproducibles usamos un **generador con semilla**: con la misma semilla, siempre salen los mismos números.

In [ ]:
rng = np.random.default_rng(seed=42)
print(rng.integers(1, 7, size=10))        # 10 lanzamientos de dado
print(rng.normal(loc=170, scale=8, size=5).round(1))  # 5 estaturas simuladas

## Ejercicios de NumPy

### Ejercicio 1
Crea un arreglo `pares` con todos los números pares desde 0 hasta 20, **incluido el 20**.

In [ ]:
pares = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.array_equal(pares, np.arange(0, 21, 2)))

### Ejercicio 2
El arreglo `temp_c` tiene la temperatura máxima de 14 días en Santiago, en grados Celsius.

1. Crea `temp_f` con las temperaturas en Fahrenheit, **sin usar ciclos**. Fórmula: F = C × 9/5 + 32.
2. Crea `dias_calurosos` con la **cantidad** de días sobre 25 °C.

In [ ]:
temp_c = np.array([22.1, 24.5, 26.3, 27.8, 21.0, 19.4, 23.7,
                   28.9, 30.2, 25.0, 24.9, 26.6, 29.1, 18.8])

temp_f = None          # TU CÓDIGO AQUÍ
dias_calurosos = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.allclose(temp_f, temp_c * 9 / 5 + 32) and dias_calurosos == (temp_c > 25).sum())

### Ejercicio 3
Con el generador `rng` y semilla 0, se simularon las notas de 30 estudiantes en 4 evaluaciones (matriz de 30 × 4).

1. Crea `prom_eval` con el promedio de **cada evaluación** (4 valores).
2. Crea `prom_est` con el promedio de **cada estudiante** (30 valores).
3. Crea `n_aprobados` con la cantidad de estudiantes cuyo promedio es mayor o igual a 4.0.

In [ ]:
rng = np.random.default_rng(seed=0)
notas_curso = rng.uniform(1.0, 7.0, size=(30, 4)).round(1)

prom_eval = None    # TU CÓDIGO AQUÍ
prom_est = None     # TU CÓDIGO AQUÍ
n_aprobados = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: np.allclose(prom_eval, notas_curso.mean(axis=0))
               and np.allclose(prom_est, notas_curso.mean(axis=1))
               and n_aprobados == (notas_curso.mean(axis=1) >= 4).sum())

---
# Parte 2. pandas: trabajar con tablas de datos

**pandas** sirve para trabajar con datos en forma de tabla, como una planilla de Excel pero con código. Tiene dos estructuras:

- **Series**: una columna con etiquetas (índice).
- **DataFrame**: una tabla con filas y columnas. Cada columna es una Series.

Por dentro, cada columna es un arreglo de NumPy, así que todo lo vectorizado de la Parte 1 funciona igual.

## 2.1 Crear un DataFrame

In [ ]:
df_mini = pd.DataFrame({
    "nombre": ["Ana", "Benja", "Carla", "Diego"],
    "comuna": ["Ñuñoa", "Maipú", "Ñuñoa", "Puente Alto"],
    "edad":   [24, 31, 28, 19],
    "horas_estudio": [6, 3, 8, 5],
})
df_mini

## 2.2 Cargar un dataset real: Titanic

Vamos a usar el dataset del **Titanic**: 891 pasajeros, con datos como clase del pasaje, sexo, edad, tarifa y si sobrevivieron. Es el ejemplo clásico para aprender análisis de datos y clasificación.

Columnas principales:

| Columna | Significado |
|---|---|
| `survived` | 1 si sobrevivió, 0 si no |
| `pclass` | Clase del pasaje: 1, 2 o 3 |
| `sex` | Sexo |
| `age` | Edad en años |
| `sibsp` | Hermanos o cónyuges a bordo |
| `parch` | Padres o hijos a bordo |
| `fare` | Tarifa pagada |
| `embarked` | Puerto de embarque (C, Q, S) |

En tus proyectos normalmente cargarás un archivo con `pd.read_csv("archivo.csv")`. Aquí lo descargamos directamente con seaborn.

In [ ]:
titanic = sns.load_dataset("titanic")
titanic = titanic[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]
titanic.head()

## 2.3 Primera exploración

Las primeras preguntas frente a cualquier dataset son siempre las mismas: ¿qué tamaño tiene?, ¿qué tipo de datos hay?, ¿faltan valores?, ¿cómo se distribuyen?

In [ ]:
print("Forma (filas, columnas):", titanic.shape)
titanic.info()

In [ ]:
# Estadísticas de las columnas numéricas
titanic.describe().round(2)

In [ ]:
# Valores faltantes por columna
titanic.isna().sum()

In [ ]:
# Conteo de categorías
titanic["pclass"].value_counts()

**Pregunta para pensar:** según `describe()`, ¿qué porcentaje de pasajeros sobrevivió? Pista: el promedio de una columna de ceros y unos es la proporción de unos.

## 2.4 Seleccionar columnas y filas

- Una columna: `df["col"]` (devuelve una Series).
- Varias columnas: `df[["col1", "col2"]]` (devuelve un DataFrame; ojo con los dobles corchetes).
- Filas por posición: `df.iloc[inicio:fin]`.
- Filas y columnas por etiqueta o condición: `df.loc[condición, columnas]`.

In [ ]:
titanic[["sex", "age", "survived"]].head(3)

In [ ]:
titanic.iloc[10:13]

In [ ]:
# Pasajeros de primera clase mayores de 60 años, mostrando solo algunas columnas
titanic.loc[(titanic["pclass"] == 1) & (titanic["age"] > 60), ["sex", "age", "fare", "survived"]]

**Regla importante para filtrar:** cada condición va entre paréntesis y se combinan con `&` (y), `|` (o) y `~` (negación). No se usan `and` ni `or` de Python.

## 2.5 Crear y transformar columnas

Las operaciones entre columnas son vectorizadas, igual que en NumPy.

In [ ]:
titanic["es_menor"] = titanic["age"] < 18
titanic["tarifa_clp"] = titanic["fare"] * 1000     # conversión inventada, solo de ejemplo
titanic[["age", "es_menor", "fare", "tarifa_clp"]].head()

In [ ]:
# map: reemplazar valores según un diccionario
titanic["puerto"] = titanic["embarked"].map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"})
titanic["puerto"].value_counts()

## 2.6 Agrupar y resumir con `groupby`

`groupby` responde preguntas del tipo "¿cuál es el promedio de X **para cada** grupo de Y?". Funciona en tres pasos: separa los datos por grupo, aplica una función a cada grupo y junta los resultados.

In [ ]:
# Tasa de supervivencia por sexo
titanic.groupby("sex")["survived"].mean().round(3)

In [ ]:
# Varias métricas a la vez
titanic.groupby("pclass").agg(
    pasajeros=("survived", "size"),
    tasa_supervivencia=("survived", "mean"),
    tarifa_promedio=("fare", "mean"),
    edad_mediana=("age", "median"),
).round(2)

In [ ]:
# Tabla cruzada: sexo x clase
pd.pivot_table(titanic, values="survived", index="sex", columns="pclass", aggfunc="mean").round(2)

## 2.7 Valores faltantes

La columna `age` tiene 177 valores faltantes (`NaN`). Hay tres estrategias comunes:

1. **Eliminar** las filas incompletas con `dropna()`. Simple, pero se pierden datos.
2. **Rellenar** con un valor fijo, como la mediana, usando `fillna()`.
3. **Rellenar por grupo**, por ejemplo con la mediana de su clase. Suele ser más preciso.

La mediana es mejor que el promedio cuando hay valores extremos.

In [ ]:
print("Filas antes:", len(titanic), "| sin faltantes en age:", len(titanic.dropna(subset=["age"])))

mediana_edad = titanic["age"].median()
edad_rellena = titanic["age"].fillna(mediana_edad)
print("Faltantes después de rellenar:", edad_rellena.isna().sum())

## 2.8 Visualizar

pandas y seaborn permiten graficar con muy poco código. Un gráfico bien elegido muestra en segundos lo que una tabla esconde.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

titanic["age"].plot(kind="hist", bins=30, ax=ejes[0], color="#1428A0")
ejes[0].set_title("Distribución de edades")
ejes[0].set_xlabel("Edad")

sns.barplot(data=titanic, x="pclass", y="survived", hue="sex", ax=ejes[1], errorbar=None)
ejes[1].set_title("Tasa de supervivencia por clase y sexo")
ejes[1].set_ylabel("Proporción que sobrevivió")

plt.tight_layout()
plt.show()

## Ejercicios de pandas

### Ejercicio 4
Crea `tasa_por_puerto`: una Series con la tasa de supervivencia de cada puerto de embarque (columna `embarked`), ordenada de mayor a menor.

In [ ]:
tasa_por_puerto = None  # TU CÓDIGO AQUÍ
tasa_por_puerto

In [ ]:
_esperado = titanic.groupby("embarked")["survived"].mean().sort_values(ascending=False)
revisar(lambda: list(tasa_por_puerto.index) == list(_esperado.index) and np.allclose(tasa_por_puerto.values, _esperado.values))

### Ejercicio 5
1. Crea la columna `familia` en `titanic`: el tamaño del grupo familiar a bordo, es decir `sibsp + parch + 1` (el 1 es el propio pasajero).
2. Crea la columna `viaja_solo`, que sea `True` cuando `familia` vale 1.
3. Crea `tasa_solo`: la tasa de supervivencia de quienes viajaban solos y de quienes no (agrupando por `viaja_solo`).

In [ ]:
# TU CÓDIGO AQUÍ


tasa_solo = None
tasa_solo

In [ ]:
revisar(lambda: (titanic["familia"] == titanic["sibsp"] + titanic["parch"] + 1).all()
               and (titanic["viaja_solo"] == (titanic["familia"] == 1)).all()
               and np.isclose(tasa_solo[True], titanic.loc[titanic["familia"] == 1, "survived"].mean()))

### Ejercicio 6
Rellena los valores faltantes de `age` con la **mediana de edad de la clase** (`pclass`) de cada pasajero y guarda el resultado en la columna `age` de `titanic`.

Pista: `titanic.groupby("pclass")["age"].transform("median")` devuelve, para cada fila, la mediana de su clase.

In [ ]:
# TU CÓDIGO AQUÍ

print("Faltantes en age:", titanic["age"].isna().sum())

In [ ]:
_orig = sns.load_dataset("titanic")
_med = _orig.groupby("pclass")["age"].transform("median")
revisar(lambda: titanic["age"].isna().sum() == 0 and np.allclose(titanic["age"], _orig["age"].fillna(_med)))

### Ejercicio 7 (análisis)
Usando `groupby` o `pivot_table`, responde: **¿qué grupo tuvo la tasa de supervivencia más baja: hombres de qué clase?** Escribe el código y anota tu conclusión en la celda de texto siguiente.

In [ ]:
# TU CÓDIGO AQUÍ

**Mi conclusión:** (escribe aquí con doble clic)

---
# Parte 3. Introducción a machine learning

## 3.1 ¿Qué es machine learning?

En la programación tradicional, una persona escribe las reglas: "si es mujer y viaja en primera clase, entonces sobrevive". En **machine learning** el programa aprende las reglas a partir de ejemplos.

Le damos al modelo:

- **X (características o features):** los datos de entrada. En el Titanic: clase, sexo, edad, tarifa, familia.
- **y (objetivo o target):** lo que queremos predecir. Aquí: `survived`.

El modelo busca una relación entre X e y. Si la encuentra, puede predecir y para pasajeros que nunca vio.

## 3.2 Tipos de problemas

| Tipo | y es... | Ejemplo | Capítulo |
|---|---|---|---|
| **Clasificación** (supervisado) | una categoría | ¿Sobrevive o no? ¿Es spam o no? | 5 |
| **Regresión** (supervisado) | un número | Precio de una casa, ventas del próximo mes | 5 |
| **Clustering** (no supervisado) | no hay y | Agrupar clientes parecidos | 6 |

Hoy resolvemos un problema de **clasificación**.

## 3.3 El flujo de trabajo

Casi todo proyecto de machine learning sigue estos pasos, y los vamos a repetir durante todo el curso:

1. Preparar los datos: limpiar, rellenar faltantes, convertir texto a números.
2. Separar X e y.
3. Dividir en entrenamiento y prueba.
4. Entrenar el modelo con `fit`.
5. Predecir con `predict`.
6. Evaluar con métricas.

## 3.4 Preparar los datos

Los modelos de scikit-learn solo entienden **números** y **no aceptan valores faltantes**. Por eso:

- `sex` (texto) se convierte en 0 y 1.
- `embarked` (tres categorías) se convierte con **one-hot encoding**: una columna de 0/1 por cada categoría. No usamos 0, 1 y 2 porque el modelo interpretaría que Q "es mayor" que C, lo que no tiene sentido.

Partimos desde una copia limpia del dataset para que esta parte funcione aunque los ejercicios anteriores no estén resueltos.

In [ ]:
datos = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()

# 1. Rellenar faltantes
datos["age"] = datos["age"].fillna(datos.groupby("pclass")["age"].transform("median"))
datos["embarked"] = datos["embarked"].fillna(datos["embarked"].mode()[0])

# 2. Convertir texto a números
datos["sex"] = datos["sex"].map({"male": 0, "female": 1})
datos = pd.get_dummies(datos, columns=["embarked"], dtype=int)

print("Faltantes totales:", datos.isna().sum().sum())
datos.head()

## 3.5 Separar X e y, y dividir en entrenamiento y prueba

Esta es la idea más importante de la clase. Si evaluamos al modelo con los mismos datos con que aprendió, es como tomar una prueba habiendo visto las respuestas: la nota no dice nada sobre si aprendió.

Por eso guardamos una parte de los datos (aquí, 20%) que el modelo **nunca ve** durante el entrenamiento, y lo evaluamos con ella.

- `test_size=0.2`: 20% para prueba.
- `random_state=42`: la división es reproducible.
- `stratify=y`: mantiene la misma proporción de sobrevivientes en ambos grupos.

In [ ]:
from sklearn.model_selection import train_test_split

X = datos.drop(columns="survived")
y = datos["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Tasa de supervivencia en train:", round(y_train.mean(), 3), "| en test:", round(y_test.mean(), 3))

## 3.6 La línea base: ¿cuánto acierta un modelo sin cerebro?

Antes de entrenar, conviene saber cuánto acierta la estrategia más tonta posible: predecir siempre la clase más frecuente ("nadie sobrevive"). Cualquier modelo útil tiene que superar esta **línea base**.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

base = DummyClassifier(strategy="most_frequent")
base.fit(X_train, y_train)
print("Exactitud de la línea base:", round(accuracy_score(y_test, base.predict(X_test)), 3))

Predecir "nadie sobrevive" acierta cerca del 62%. Eso no es un buen modelo; solo refleja que la mayoría murió. Recuerda este número.

## 3.7 Entrenar el primer modelo: regresión logística

Toda la API de scikit-learn funciona igual, sea cual sea el modelo:

```python
modelo = AlgunModelo(parametros)
modelo.fit(X_train, y_train)      # aprender
predicciones = modelo.predict(X_test)  # predecir
```

La **regresión logística**, a pesar de su nombre, es un modelo de clasificación. Calcula la probabilidad de que y sea 1 combinando las características con pesos que aprende de los datos. La veremos en detalle en el capítulo 5.

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
print("Primeras 10 predicciones:", pred[:10])
print("Primeros 10 valores reales:", y_test.values[:10])
print("Exactitud en prueba:", round(accuracy_score(y_test, pred), 3))

## 3.8 Evaluar más allá de la exactitud

La **exactitud** (accuracy) es la proporción de aciertos. Es útil, pero no dice **en qué** se equivoca el modelo. Para eso usamos la **matriz de confusión**:

|  | Predice 0 | Predice 1 |
|---|---|---|
| **Real 0** | Verdaderos negativos | Falsos positivos |
| **Real 1** | Falsos negativos | Verdaderos positivos |

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["No sobrevive", "Sobrevive"], cmap="Blues")
plt.title("Matriz de confusión: regresión logística")
plt.show()

print(classification_report(y_test, pred, target_names=["No sobrevive", "Sobrevive"]))

En el reporte:

- **Precision** de "Sobrevive": de los que el modelo dijo que sobrevivían, qué proporción realmente sobrevivió.
- **Recall** de "Sobrevive": de los que realmente sobrevivieron, qué proporción detectó el modelo.

Profundizaremos en estas métricas en el capítulo 5.

## 3.9 Predecir para pasajeros nuevos

Un modelo entrenado sirve para casos que no existían en los datos. `predict_proba` entrega la probabilidad de cada clase.

In [ ]:
nuevos = pd.DataFrame({
    "pclass":     [1, 3],
    "sex":        [1, 0],      # 1 = mujer, 0 = hombre
    "age":        [30, 25],
    "sibsp":      [0, 0],
    "parch":      [0, 0],
    "fare":       [80.0, 7.5],
    "embarked_C": [1, 0],
    "embarked_Q": [0, 0],
    "embarked_S": [0, 1],
})[X.columns]   # mismas columnas y en el mismo orden que X

prob = modelo.predict_proba(nuevos)[:, 1]
for desc, p in zip(["Mujer, 30 años, primera clase", "Hombre, 25 años, tercera clase"], prob):
    print(f"{desc}: {p:.1%} de probabilidad de sobrevivir")

## 3.10 Otro modelo: árbol de decisión y sobreajuste

Un **árbol de decisión** aprende una serie de preguntas del tipo "¿sex = mujer?", "¿pclass ≤ 2?". Es fácil de interpretar, pero tiene un riesgo: si lo dejamos crecer sin límite, **memoriza** los datos de entrenamiento en vez de aprender patrones generales. A eso se le llama **sobreajuste** (overfitting).

El parámetro `max_depth` limita cuántas preguntas seguidas puede hacer. Observa qué pasa con la exactitud en entrenamiento y en prueba al aumentarlo:

In [ ]:
from sklearn.tree import DecisionTreeClassifier

profundidades = range(1, 16)
exac_train, exac_test = [], []
for d in profundidades:
    arbol = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    exac_train.append(arbol.score(X_train, y_train))
    exac_test.append(arbol.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(profundidades, exac_train, marker="o", label="Entrenamiento")
plt.plot(profundidades, exac_test, marker="o", label="Prueba")
plt.xlabel("max_depth (profundidad del árbol)")
plt.ylabel("Exactitud")
plt.title("Sobreajuste: el árbol memoriza a medida que crece")
plt.legend(); plt.grid(alpha=0.3); plt.show()

A medida que el árbol crece, la exactitud en entrenamiento se acerca al 100%, pero en prueba se estanca o baja. **El modelo que nos interesa es el que mejor generaliza**, es decir, el que mejor funciona en datos que no vio. Por eso siempre evaluamos con el conjunto de prueba.

Un árbol pequeño se puede dibujar y leer como un diagrama de reglas:

In [ ]:
from sklearn.tree import plot_tree

arbol_chico = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(12, 5))
plot_tree(arbol_chico, feature_names=X.columns, class_names=["No", "Sí"], filled=True, rounded=True)
plt.show()

## Ejercicios de machine learning

### Ejercicio 8
Entrena un `DecisionTreeClassifier` con `max_depth=4` y `random_state=42` usando `X_train` e `y_train`. Guárdalo en `arbol4` y calcula su exactitud en prueba en `exac_arbol4`.

¿Supera a la regresión logística? ¿Y a la línea base?

In [ ]:
arbol4 = None       # TU CÓDIGO AQUÍ
exac_arbol4 = None  # TU CÓDIGO AQUÍ
print("Exactitud árbol (max_depth=4):", exac_arbol4)

In [ ]:
_ref = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train).score(X_test, y_test)
revisar(lambda: np.isclose(exac_arbol4, _ref))

### Ejercicio 9
¿Mejora el modelo si le damos más información?

1. Copia `datos` en `datos2` y agrégale la columna `familia` (`sibsp + parch + 1`).
2. Separa `X2` e `y2`, y divide con **los mismos parámetros** de antes (`test_size=0.2`, `random_state=42`, `stratify=y2`).
3. Entrena una `LogisticRegression(max_iter=1000)` y guarda su exactitud en prueba en `exac_familia`.
4. Compárala con la exactitud del modelo original.

In [ ]:
# TU CÓDIGO AQUÍ


exac_familia = None
print("Exactitud con 'familia':", exac_familia)

In [ ]:
_d2 = datos.copy(); _d2["familia"] = _d2["sibsp"] + _d2["parch"] + 1
_X2, _y2 = _d2.drop(columns="survived"), _d2["survived"]
_a, _b, _c, _e = train_test_split(_X2, _y2, test_size=0.2, random_state=42, stratify=_y2)
_ref = LogisticRegression(max_iter=1000).fit(_a, _c).score(_b, _e)
revisar(lambda: np.isclose(exac_familia, _ref))

### Ejercicio 10 (análisis)
Ejecuta la siguiente celda: muestra qué importancia le da el árbol de profundidad 4 a cada característica. ¿Cuáles son las tres más importantes? ¿Coincide con lo que encontraste en la Parte 2 con `groupby`?

In [ ]:
try:
    importancias = pd.Series(arbol4.feature_importances_, index=X.columns).sort_values()
    importancias.plot(kind="barh", color="#1428A0", figsize=(7, 4), title="Importancia de cada característica")
    plt.show()
except AttributeError:
    print("Primero resuelve el Ejercicio 8.")

**Mi conclusión:** (escribe aquí con doble clic)

---
# Parte 4. Desafío final y guardar en GitHub

## Desafío: clasificar especies de pingüinos

El dataset **penguins** tiene medidas de 344 pingüinos de tres especies (Adelie, Chinstrap y Gentoo) en islas de la Antártica. Aplica el flujo completo de la clase para predecir la especie (`species`) a partir de sus medidas.

Pasos sugeridos:

1. Explora el dataset: forma, tipos, faltantes, `value_counts` de la especie.
2. Elimina las filas con faltantes (son pocas).
3. Usa como X las columnas numéricas: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`.
4. Divide en entrenamiento y prueba (20%, `random_state=42`, estratificado).
5. Calcula la línea base, entrena una regresión logística y un árbol de decisión, y compara sus exactitudes.
6. Muestra la matriz de confusión del mejor modelo.

**Extra:** agrega `island` y `sex` como características con one-hot encoding. ¿Mejora el modelo?

In [ ]:
pinguinos = sns.load_dataset("penguins")
pinguinos.head()

In [ ]:
# TU CÓDIGO AQUÍ

## Guardar tu trabajo en GitHub

Con lo que vimos el jueves, guarda este notebook en tu repositorio:

1. *Archivo > Guardar una copia en GitHub*.
2. Autoriza a Colab a acceder a tu cuenta de GitHub, si te lo pide.
3. Elige tu repositorio y la rama `main`.
4. Escribe un mensaje de commit descriptivo, por ejemplo: `Clase 2: NumPy, pandas e intro a ML`.
5. Marca la opción *Incluir un enlace a Colab* y confirma.

Repositorio del curso: https://github.com/davidlealo/sic_ia_2026

---
# Soluciones

Intenta resolver cada ejercicio antes de mirar aquí. Si ejecutas estas celdas, se sobrescriben tus variables con la solución.

### Ejercicio 1

In [ ]:
pares = np.arange(0, 21, 2)
revisar(lambda: np.array_equal(pares, np.arange(0, 21, 2)))

### Ejercicio 2

In [ ]:
temp_f = temp_c * 9 / 5 + 32
dias_calurosos = (temp_c > 25).sum()
print(temp_f.round(1), "| días calurosos:", dias_calurosos)

### Ejercicio 3

In [ ]:
prom_eval = notas_curso.mean(axis=0)
prom_est = notas_curso.mean(axis=1)
n_aprobados = (prom_est >= 4.0).sum()
print(prom_eval.round(2), "| aprobados:", n_aprobados)

### Ejercicio 4

In [ ]:
tasa_por_puerto = titanic.groupby("embarked")["survived"].mean().sort_values(ascending=False)
tasa_por_puerto

### Ejercicio 5

In [ ]:
titanic["familia"] = titanic["sibsp"] + titanic["parch"] + 1
titanic["viaja_solo"] = titanic["familia"] == 1
tasa_solo = titanic.groupby("viaja_solo")["survived"].mean()
tasa_solo

### Ejercicio 6

In [ ]:
titanic["age"] = titanic["age"].fillna(titanic.groupby("pclass")["age"].transform("median"))
print("Faltantes en age:", titanic["age"].isna().sum())

### Ejercicio 7
Los hombres de tercera clase tuvieron la tasa más baja, cercana al 14%.

In [ ]:
pd.pivot_table(titanic, values="survived", index="sex", columns="pclass", aggfunc="mean").round(3)

### Ejercicio 8

In [ ]:
arbol4 = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
exac_arbol4 = arbol4.score(X_test, y_test)
print("Árbol:", round(exac_arbol4, 3), "| Logística:", round(accuracy_score(y_test, pred), 3), "| Base:", round(base.score(X_test, y_test), 3))

### Ejercicio 9

In [ ]:
datos2 = datos.copy()
datos2["familia"] = datos2["sibsp"] + datos2["parch"] + 1
X2 = datos2.drop(columns="survived")
y2 = datos2["survived"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
exac_familia = LogisticRegression(max_iter=1000).fit(X2_train, y2_train).score(X2_test, y2_test)
print("Con familia:", round(exac_familia, 3), "| Original:", round(accuracy_score(y_test, pred), 3))

Una característica nueva no siempre mejora el modelo. `familia` es la suma de `sibsp` y `parch`, que ya estaban en X, así que aporta poca información nueva a un modelo lineal.

### Ejercicio 10
Las características más importantes suelen ser `sex`, `pclass` y `age` (o `fare`). Coincide con el análisis de la Parte 2: el sexo y la clase fueron los factores que más separaron a sobrevivientes de no sobrevivientes.

### Desafío final

In [ ]:
p = sns.load_dataset("penguins").dropna()
print(p["species"].value_counts())

cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
Xp, yp = p[cols], p["species"]
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(Xp, yp, test_size=0.2, random_state=42, stratify=yp)

modelos = {
    "Línea base": DummyClassifier(strategy="most_frequent"),
    "Regresión logística": LogisticRegression(max_iter=5000),
    "Árbol de decisión": DecisionTreeClassifier(max_depth=4, random_state=42),
}
for nombre, m in modelos.items():
    m.fit(Xp_tr, yp_tr)
    print(f"{nombre}: {m.score(Xp_te, yp_te):.3f}")

ConfusionMatrixDisplay.from_estimator(modelos["Regresión logística"], Xp_te, yp_te, cmap="Blues")
plt.title("Pingüinos: regresión logística")
plt.show()

In [ ]:
# Extra: agregar isla y sexo con one-hot encoding
Xp2 = pd.get_dummies(p[cols + ["island", "sex"]], columns=["island", "sex"], dtype=int)
Xa, Xb, ya, yb = train_test_split(Xp2, yp, test_size=0.2, random_state=42, stratify=yp)
print("Logística con isla y sexo:", round(LogisticRegression(max_iter=5000).fit(Xa, ya).score(Xb, yb), 3))